# Tailor — Worked Example

A 10-minute end-to-end walkthrough for health researchers and research-software engineers. Run every cell top-to-bottom and you'll see:

1. How the router, a biosensor child, and the vault layer wire up — the same sequence `tailor serve` runs on startup.
2. A Tier-1 analytical call (server-computed summary) against synthetic run data.
3. The audit row the router wrote to SQLite for that call.
4. The Tier-2 consent gate firing, the structured gate response the LLM sees, and the `approve_consent_*` follow-up that unblocks it.
5. A vault theme written and read back both as a vault tool response and as a plain Obsidian-compatible markdown file on disk.

**You will not need:** a Strava account, OAuth credentials, a network connection, a real Obsidian vault, or an LLM. Everything runs locally against synthetic data inside a temporary directory that is discarded when the kernel shuts down.

**Takeaway:** data-minimization (the tier model), provenance (the `_meta` block), reproducibility (the audit log), and cross-session memory (the vault) all fire without the analyst or the LLM client doing anything domain-specific. The governance is in the server, not in the client.

> **Setup.** From the repo root: `pip install -e ".[dev]"` and `pip install jupyter` (or open in VS Code's Jupyter support). The only runtime dependencies of the cells below are what the package already pulls in.


## 1. Wire the router, a child, and the vault

This cell is equivalent to `cmd_serve()` in `tailor/__main__.py` with two differences: paths point at a tempdir, and we skip the stdio transport so you can inspect return values from Python instead of through MCP JSON-RPC.


In [ ]:
import asyncio
import json
import sqlite3
import tempfile
from pathlib import Path

from tailor.children.running import RunningChild
from tailor.demo.sample_data import (
    SAMPLE_ACTIVITY_ID,
    generate_sample_activity,
    generate_sample_streams,
)
from tailor.framework.router import RouterMCP
from tailor.framework.vault import VaultLayer, VaultWriter

# Everything lives in a tempdir — nothing touches your real config.
workdir = Path(tempfile.mkdtemp(prefix="tailor-worked-example-"))
config_dir = workdir / "config"; config_dir.mkdir()
data_dir = workdir / "data"; data_dir.mkdir()
vault_dir = workdir / "vault"; vault_dir.mkdir()

# Minimum RunningChild expects to find. The 'demo' OAuth tokens never get
# used — we inject the synthetic activity directly into SQLite below.
(config_dir / "user_config.json").write_text(
    json.dumps({"max_hr": 185, "resting_hr": 52})
)
(config_dir / "tokens.json").write_text(json.dumps({
    "client_id": "demo", "client_secret": "demo",
    "access_token": "demo", "refresh_token": "demo", "expires_at": 0,
}))

print(f"Workspace: {workdir}")

In [ ]:
router = RouterMCP(name="worked-example", data_dir=data_dir)

# Register the biosensor child. In production this reads Strava; here we
# pre-seed its local SQLite cache with one synthetic 60-minute run.
running = RunningChild(config_dir=config_dir, data_dir=data_dir)
router.register_child(running)
running._storage.save_activity(SAMPLE_ACTIVITY_ID, generate_sample_activity())
running._storage.save_streams(SAMPLE_ACTIVITY_ID, generate_sample_streams())

# Wire the vault layer. It's framework-level infrastructure, not a ChildMCP
# — the router skips consent/cost/circuit gates for vault tools because
# their payload is analyst notes, not participant biometric data.
vault_writer = VaultWriter(
    vault_path=vault_dir,
    data_dir=data_dir,
    vaultable_tools=set(running.vaultable_tools),
    max_hr=185,
)
router.register_post_execute_hook(vault_writer)
router.register_vault_layer(VaultLayer(
    vault_path=vault_dir,
    vault_writer=vault_writer,
    backfill_config={
        "list_tool": "strava_list_runs",
        "report_tool": "strava_run_report",
    },
))

print(f"Registered domains: {router.registered_domains}")
print(f"Tool count:         {len(router.registered_tools)}")

## 2. A Tier-1 call — server-computed summary, no gate

Tier 1 is where most analytical questions are answered. The LLM sees a
compact report — drift, decoupling, EF, zones, splits — computed from
per-second streams that never leave this machine.

The helper below calls `router._dispatch` directly. That's what an MCP
client invokes over stdio; we skip the transport so the notebook can
show you the return value instead of a JSON-RPC envelope.


In [ ]:
async def call(tool_name: str, params: dict | None = None) -> dict:
    """Dispatch a tool the way an MCP client would; return the parsed result."""
    chunks = await router._dispatch(tool_name, params or {})
    return json.loads(chunks[0].text)

report = asyncio.run(call("strava_run_report", {"activity_id": SAMPLE_ACTIVITY_ID}))

print("Top-level keys:", sorted(report.keys()))
print()
print(f"Activity:      {report.get('activity_name')}")
print(f"Data points:   {report['data_points']}")
print(f"Avg HR:        {report.get('average_heartrate')} bpm")
print(f"HR drift:      {report['hr_drift']['drift_pct']:.2f} %")
print(f"Decoupling:    {report['decoupling']['decoupling_pct']:.2f} %")
print(f"EF:            {report['efficiency_factor']['ef']:.3f}")
print(f"HR zones:      {report['hr_zones']}")
print()
print("_meta (provenance stamp on every result):")
for k, v in report["_meta"].items():
    print(f"  {k}: {v}")

Note three things about the output:

- **No raw streams.** The per-second heart-rate and velocity arrays stayed on this machine; only summary numbers crossed the router boundary.
- **`_meta` is unconditional.** Every successful result carries `package_version`, `tool_name`, and UTC `called_at`. Six months from now, any number you quote in a paper is traceable to the exact code version that produced it.
- **`tier: 1`.** The value came back without triggering consent or cost gates. Those only fire on Tier-2+ tools, by design.


## 3. The audit log

The dispatch above appended exactly one row to `audit.db`. Blocked calls
(param-invalid, consent-blocked, cost-gated, circuit-open, execution
error) append rows too, with an outcome string naming the reason. The
audit log is the reproducibility backbone — it is the answer to "what
did an LLM analyst see, when, at what tier, on which subject?"


In [ ]:
with sqlite3.connect(str(data_dir / "audit.db")) as conn:
    rows = conn.execute(
        "SELECT timestamp, domain, tool_name, tier, outcome, duration_ms, token_estimate "
        "FROM audit_log ORDER BY id DESC LIMIT 5"
    ).fetchall()

hdr = f"{'timestamp':<28} {'domain':<8} {'tool':<32} {'tier':<4} {'outcome':<14} {'ms':<5} tokens"
print(hdr)
print("-" * len(hdr))
for ts, dom, tool, tier, outcome, ms, tokens in rows:
    print(f"{ts:<28} {dom:<8} {tool:<32} {tier:<4} {outcome:<14} {ms:<5} {tokens}")

## 4. The consent gate — Tier-2

Tier-2 tools return downsampled streams for visualization. Still a
data-minimization win over raw per-second streams, but now the LLM is
about to see biometric values at 5–30-second resolution. The router
blocks the first call with a **structured response** the client must
surface to the user as a yes/no prompt; only an explicit
`approve_consent_<domain>` call unblocks the session.

The gate response is a JSON object with individually checkable
`must_do` / `must_not_do` / `on_ambiguous_reply` fields
([ADR 0004](../adr/0004-structured-llm-instruction.md)) rather than a
free-text paragraph — so a reviewer can audit whether the client
actually surfaced every obligation.


In [ ]:
# Attempt 1: consent has not been granted, so the router short-circuits.
gate = asyncio.run(call(
    "strava_downsampled_streams",
    {"activity_id": SAMPLE_ACTIVITY_ID, "interval_seconds": 10},
))
print(f"[1] gate fired: {gate.get('gate')}")
print(f"    requested:  {gate.get('data_requested_this_call')}")
print(f"    scope:      {gate['scope']['duration_human']}")
print(f"    user prompt (client must surface verbatim):")
print(f"      {gate['user_prompt']}")

# Approve.
approval = asyncio.run(call("approve_consent_running"))
print(f"\n[2] {approval['message']}")

# Attempt 2: same call now succeeds.
downsampled = asyncio.run(call(
    "strava_downsampled_streams",
    {"activity_id": SAMPLE_ACTIVITY_ID, "interval_seconds": 10},
))
print(f"\n[3] original:     {downsampled['original_points']} points")
print(f"    downsampled:  {downsampled['downsampled_points']} points "
      f"({downsampled['reduction_pct']}% reduction)")
print(f"    streams sent: {downsampled['streams_included']}")
print(f"    tier:         {downsampled['_meta']['tier']}")
print(f"    tokens:       {downsampled['_meta']['tokens_this_call']}")

Inspect the audit log again — three new rows, one per attempt, each
with an outcome recording exactly what happened:


In [ ]:
with sqlite3.connect(str(data_dir / "audit.db")) as conn:
    rows = conn.execute(
        "SELECT timestamp, tool_name, tier, outcome "
        "FROM audit_log ORDER BY id DESC LIMIT 3"
    ).fetchall()

for ts, tool, tier, outcome in rows:
    print(f"{ts}  tier={tier}  {tool:<32} -> {outcome}")

## 5. The vault layer — cross-session analytical memory

The vault layer persists the *reasoning* tier: **themes** (persistent
hypotheses an analyst is tracking across runs), **moments** (one-off
observations), and evidence logs that append, never overwrite.

Markdown files are the source of truth — the `vault_path` directory is
an Obsidian vault a human can open, browse, and hand-edit. `vault.db`
is a query-optimization index over those files; if it disappears, a
`vault_rescan` call rebuilds it from the markdown.


In [ ]:
# Write a research theme — a persistent hypothesis the analyst wants
# to track across sessions. Evidence blocks are append-only; calling
# vault_upsert_theme again on the same slug would APPEND a new block
# rather than overwrite this one.
asyncio.run(call("vault_upsert_theme", {
    "slug": "aerobic-drift-recovery-runs",
    "title": "Aerobic drift on recovery runs",
    "hypothesis": (
        "HR drift above 3% on nominal recovery pace suggests incomplete "
        "recovery from the prior session's load."
    ),
    "evidence": (
        f"Activity {SAMPLE_ACTIVITY_ID}: drift {report['hr_drift']['drift_pct']:.1f}% "
        f"at avg HR {report.get('average_heartrate')} despite easy pace."
    ),
    "status": "open",
    "confidence": "low",
    "linked_runs": [SAMPLE_ACTIVITY_ID],
    "tags": ["aerobic-base", "recovery"],
    # Evidence provenance — stamps where this observation came from.
    "evidence_source_tier": 1,
    "evidence_source_tool": "strava_run_report",
    "evidence_source_domain": "running",
    "evidence_verification": "computed",
}))

# Read themes back through the vault's own tool.
themes = asyncio.run(call("vault_list_themes", {"status": "open"}))
print(f"Open themes: {len(themes['themes'])}")
for t in themes["themes"]:
    print(f"  - {t['slug']}  "
          f"(confidence={t.get('confidence')}, "
          f"linked_runs={t.get('linked_run_count', 0)})")

In [ ]:
# Read the raw markdown file — this is exactly what a researcher sees
# when they open the vault in Obsidian. No special format, no lock-in.
theme_file = next(vault_dir.rglob("aerobic-drift-recovery-runs.md"))
print(f"--- {theme_file.relative_to(vault_dir)} ---")
print(theme_file.read_text())

## 6. Cleanup

`router.close()` releases the SQLite WAL locks (required on Windows
before the tempdir can be removed). In a real server this runs
automatically in `__main__.cmd_serve`'s `finally` block.


In [ ]:
router.close()
print(f"Closed. Workspace still on disk at: {workdir}")
print("(Will be removed next time your OS cleans /tmp.)")

## Where to next

- **Wrap your own data source.** Copy [`src/tailor/children/template/`](../../src/tailor/children/template/) and fill in the four abstract methods. The framework's pipeline (validation, circuit breaker, consent, cost, PHI seam, audit, vault writes) applies to your child unchanged.
- **Swap in a real PHI policy.** [ADR 0003](../adr/0003-phi-scrubber-seam.md) covers the `PHIScrubber` extension seam — a no-op default that your institutional subclass replaces once an IRB-approved policy exists.
- **Read the architecture decisions.** [docs/adr/](../adr/) covers the five load-bearing choices: audit-log backbone, `subject_id` scoping, PHI-scrubber seam, structured LLM instructions, and pre-estimation cost gating.
- **See what is explicitly deferred.** [ROADMAP.md](../../ROADMAP.md) — including CGM/sleep/ECG/EDF/FHIR children, deterministic replay, and full provenance hashing.
